# 13.3 - Model Selection & Routing

**Module 13 - Cost & Performance** | Netsetos GenAI Engineering

Most queries don't need a frontier model, yet sending everything to the most expensive tier marks up the easy majority of your traffic 20-100x for a quality gap you can't even measure. This notebook builds routing tooling from the ground up: a price-performance analyzer, an eval-driven model selector, rule-based and classifier routers, a cheap-first cascade, a gateway with fallbacks, and a per-route drift monitor. Every cell is keyless and runnable - the numbers are illustrative so you can re-run with your own to build intuition.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

This cell prepares the environment for the routing simulations that follow. Every code cell in this notebook is self-contained and keyless - the numbers are illustrative, so nothing here calls a live API.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A lightweight environment-prep cell. It carries a commented-out `pip install anthropic` line for fresh environments like Colab, plus a note that the lesson uses fixed, illustrative values throughout rather than live prices or random draws. Nothing executes that produces output.

**How the code works - step by step**
- The commented `pip install anthropic -q` line is there to uncomment only if you later extend these examples to real API calls.
- The reproducibility comment flags that all cost, quality, and confidence figures downstream are hand-picked illustrative constants, not fetched live.
- **In one line:** environment scaffolding - no dependencies are actually installed and nothing runs.

**What you'll see:** (no output - environment prep)

## 1 - The price-performance spread

This cell prices one workload across four model tiers to show why "one model for everything" is the wrong default. It compares running everything on the frontier, everything on the small model, and matching each request to its difficulty.

In [ ]:
TIERS = {                       # illustrative $/1M tokens + eval-score; NOT live prices
    "frontier": {"in": 5.00, "out": 25.00, "q": 0.95},   # opus / GPT-5-class
    "mid":      {"in": 3.00, "out": 15.00, "q": 0.90},   # sonnet-class
    "small":    {"in": 1.00, "out":  5.00, "q": 0.82},   # haiku-class
    "nano":     {"in": 0.25, "out":  1.50, "q": 0.72},   # flash-lite / nano-class
}
def cost(tier, tin, tout):      # per-request cost in USD
    return (tin * TIERS[tier]["in"] + tout * TIERS[tier]["out"]) / 1_000_000
# Why "one model for everything" is the wrong default: the price-performance SPREAD.
TIN, TOUT, REQ = 1000, 300, 10000        # one workload: 1000 in / 300 out, 10k requests/day
print("Per-request cost by tier (in {} / out {} tokens):".format(TIN, TOUT))
for t in TIERS:
    print("  {:<9} ${:.4f}/req   quality {:.2f}".format(t, cost(t, TIN, TOUT), TIERS[t]["q"]))
print()
all_frontier = cost("frontier", TIN, TOUT) * REQ
all_small    = cost("small", TIN, TOUT) * REQ
# matched: send the easy 70% to small, the medium 20% to mid, the hard 10% to frontier
matched = (0.70*cost("small",TIN,TOUT) + 0.20*cost("mid",TIN,TOUT) + 0.10*cost("frontier",TIN,TOUT)) * REQ
print("Spread: frontier input is {:.0f}x the nano input price - but the quality gap is only {:.2f}.".format(
    TIERS["frontier"]["in"]/TIERS["nano"]["in"], TIERS["frontier"]["q"]-TIERS["nano"]["q"]))
print("At {:,} req/day:".format(REQ))
print("  everything on frontier: ${:,.2f}/day   (safe, but you overpay on every easy query)".format(all_frontier))
print("  everything on small:    ${:,.2f}/day   ({:.0f}x cheaper - but quality drops on the hard ones)".format(all_small, all_frontier/all_small))
print("  matched to difficulty:  ${:,.2f}/day   ({:.1f}x cheaper, frontier kept for the hard 10%)".format(matched, all_frontier/matched))
print()
print("Most queries do not need a frontier model. Matching the model to the task is the lever this lesson pulls.")

# Output:
# Per-request cost by tier (in 1000 / out 300 tokens):
#   frontier  $0.0125/req   quality 0.95
#   mid       $0.0075/req   quality 0.90
#   small     $0.0025/req   quality 0.82
#   nano      $0.0007/req   quality 0.72
#
# Spread: frontier input is 20x the nano input price - but the quality gap is only 0.23.
# At 10,000 req/day:
#   everything on frontier: $125.00/day   (safe, but you overpay on every easy query)
#   everything on small:    $25.00/day   (5x cheaper - but quality drops on the hard ones)
#   matched to difficulty:  $45.00/day   (2.8x cheaper, frontier kept for the hard 10%)
#
# Most queries do not need a frontier model. Matching the model to the task is the lever this lesson pulls.

**Explanation**

A price-performance analyzer built on a shared `TIERS` table (dollars per million tokens plus an eval-quality score for each tier) and a `cost()` helper. It computes the per-request cost of each tier, then the daily cost of three strategies over 10,000 requests to expose the 20x price spread against a tiny quality gap.

**How the code works - step by step**
- `TIERS` defines four tiers (frontier, mid, small, nano) with input/output prices and a quality score `q` - this same table is reused in every later cell.
- `cost(tier, tin, tout)` returns the per-request USD cost for a given input/output token count.
- Fixed workload constants `TIN=1000, TOUT=300, REQ=10000` describe one day of traffic.
- It prints each tier's per-request cost and quality, then computes `all_frontier`, `all_small`, and `matched` (70% small / 20% mid / 10% frontier) daily costs.
- **In one line:** it quantifies how matching the model to task difficulty beats defaulting everything to the frontier.

**What you'll see:** A per-tier cost/quality table, then daily totals: everything on frontier ~$125/day, everything on small ~$25/day (5x cheaper), and matched-to-difficulty ~$45/day (2.8x cheaper while keeping frontier for the hard 10%).

## 2 - Eval-driven model selection

This cell turns model choice into a measurement problem. Given per-task eval scores for each tier and a quality bar, it picks the cheapest model that clears the bar rather than defaulting to the most impressive one.

In [ ]:
TIERS = {                       # illustrative $/1M tokens + eval-score; NOT live prices
    "frontier": {"in": 5.00, "out": 25.00, "q": 0.95},   # opus / GPT-5-class
    "mid":      {"in": 3.00, "out": 15.00, "q": 0.90},   # sonnet-class
    "small":    {"in": 1.00, "out":  5.00, "q": 0.82},   # haiku-class
    "nano":     {"in": 0.25, "out":  1.50, "q": 0.72},   # flash-lite / nano-class
}
def cost(tier, tin, tout):      # per-request cost in USD
    return (tin * TIERS[tier]["in"] + tout * TIERS[tier]["out"]) / 1_000_000
# Selection = measure, do not guess. Build a small eval set per TASK, score each model, pick the
# CHEAPEST model that clears your quality bar (the RouterBench idea).
QUALITY_BAR = 0.85
TIN, TOUT = 1000, 300
# measured eval scores for THIS task (illustrative; from running a 20-example eval set per model)
measured = {"nano": 0.71, "small": 0.86, "mid": 0.91, "frontier": 0.94}
print("Task eval (quality bar = {:.2f}) - cheapest model that CLEARS the bar wins:".format(QUALITY_BAR))
pick = None
for t in ["nano", "small", "mid", "frontier"]:   # cheapest -> most expensive
    q, c = measured[t], cost(t, TIN, TOUT)
    ok = q >= QUALITY_BAR
    if ok and pick is None: pick = t
    print("  {:<9} quality {:.2f}  ${:.4f}/req  -> {}".format(t, q, c, "CLEARS the bar" if ok else "below the bar"))
print()
front_c, pick_c = cost("frontier", TIN, TOUT), cost(pick, TIN, TOUT)
print("Pick: {} at quality {:.2f}, ${:.4f}/req.".format(pick, measured[pick], pick_c))
print("  vs defaulting to frontier ({:.2f}, ${:.4f}): {:.0f}x cheaper, and the bar is still met.".format(
    measured["frontier"], front_c, front_c/pick_c))
print("  frontier's extra {:.2f} quality is not worth {:.0f}x the price for THIS task.".format(
    measured["frontier"]-measured[pick], front_c/pick_c))
print("Selection is per-task: run the eval, plot quality vs cost, keep the cheapest that clears the bar.")

# Output:
# Task eval (quality bar = 0.85) - cheapest model that CLEARS the bar wins:
#   nano      quality 0.71  $0.0007/req  -> below the bar
#   small     quality 0.86  $0.0025/req  -> CLEARS the bar
#   mid       quality 0.91  $0.0075/req  -> CLEARS the bar
#   frontier  quality 0.94  $0.0125/req  -> CLEARS the bar
#
# Pick: small at quality 0.86, $0.0025/req.
#   vs defaulting to frontier (0.94, $0.0125): 5x cheaper, and the bar is still met.
#   frontier's extra 0.08 quality is not worth 5x the price for THIS task.
# Selection is per-task: run the eval, plot quality vs cost, keep the cheapest that clears the bar.

**Explanation**

An eval-driven selector implementing the RouterBench idea in miniature. It walks the tiers from cheapest to most expensive, checks each measured quality score against a `QUALITY_BAR`, and locks in the first one that clears it - then quantifies how much cheaper that pick is than defaulting to frontier.

**How the code works - step by step**
- Reuses the `TIERS` table and `cost()` helper from Concept 1.
- `QUALITY_BAR = 0.85` is the bar this task must clear; `measured` holds the illustrative per-model eval scores for this specific task.
- The loop iterates nano -> frontier (cheapest first) and sets `pick` to the first tier whose quality meets the bar.
- It then compares the pick's cost against the frontier's, printing the cost multiple saved and noting the frontier's extra quality isn't worth the price for this task.
- **In one line:** measure a per-task eval, then keep the cheapest model that clears the bar.

**What you'll see:** A table marking each tier as clearing or below the 0.85 bar (nano fails), then the pick - the small model at quality 0.86 - which is 5x cheaper than frontier while still meeting the bar.

## 3 - Rule-based routing

This cell builds the cheapest possible router: a lookup table of keyword and length rules evaluated before any model is called. It routes a batch of queries and compares the cost to sending everything to the frontier.

In [ ]:
TIERS = {                       # illustrative $/1M tokens + eval-score; NOT live prices
    "frontier": {"in": 5.00, "out": 25.00, "q": 0.95},   # opus / GPT-5-class
    "mid":      {"in": 3.00, "out": 15.00, "q": 0.90},   # sonnet-class
    "small":    {"in": 1.00, "out":  5.00, "q": 0.82},   # haiku-class
    "nano":     {"in": 0.25, "out":  1.50, "q": 0.72},   # flash-lite / nano-class
}
def cost(tier, tin, tout):      # per-request cost in USD
    return (tin * TIERS[tier]["in"] + tout * TIERS[tier]["out"]) / 1_000_000
# Rule-based routing: the CHEAPEST router (~0 ms). Route on simple query signals - task type, length -
# with a lookup table. Pre-request rules catch the obvious cases for free.
TOUT = 300
def route_rules(query, tin):
    q = query.lower()
    if tin > 2000:                                   return "frontier"  # very long context -> strongest
    if any(k in q for k in ("prove", "design", "debug", "analyze")):  return "mid"   # reasoning-ish
    if any(k in q for k in ("classify", "extract", "format", "tag")): return "small" # mechanical
    return "small"                                   # default: assume easy until shown otherwise
queries = [
    ("Classify this ticket as bug or feature", 120),
    ("Extract the invoice total and date", 90),
    ("Design a schema for multi-tenant billing", 300),
    ("Summarize this 4000-token contract", 4000),
    ("Tag the sentiment of this review", 60),
]
print("Rule-based routing (0 ms overhead - just string + length checks):")
routed = 0.0
for text, tin in queries:
    t = route_rules(text, tin)
    c = cost(t, tin, TOUT)
    routed += c
    print("  {:<9} <- {}".format(t, text[:38]))
front = sum(cost("frontier", tin, TOUT) for _, tin in queries)
print()
print("Batch cost: routed ${:.4f} vs all-frontier ${:.4f}  = {:.1f}x cheaper, decided before the call.".format(
    routed, front, front/routed))
print("Rules are brittle (a short query can still be hard), so they are the FLOOR, not the whole story.")

# Output:
# Rule-based routing (0 ms overhead - just string + length checks):
#   small     <- Classify this ticket as bug or feature
#   small     <- Extract the invoice total and date
#   mid       <- Design a schema for multi-tenant billi
#   frontier  <- Summarize this 4000-token contract
#   small     <- Tag the sentiment of this review
#
# Batch cost: routed $0.0377 vs all-frontier $0.0604  = 1.6x cheaper, decided before the call.
# Rules are brittle (a short query can still be hard), so they are the FLOOR, not the whole story.

**Explanation**

A pre-request rule router - essentially a triage clipboard. The `route_rules()` function reads cheap surface signals (input length, task-type keywords) and returns a tier with zero model calls, so overhead is effectively 0 ms. It then routes a sample batch and totals the cost against an all-frontier baseline.

**How the code works - step by step**
- `route_rules(query, tin)` applies ordered checks: very long context (>2000 tokens) -> frontier, reasoning keywords (prove/design/debug/analyze) -> mid, mechanical keywords (classify/extract/format/tag) -> small, otherwise default to small.
- `queries` is a batch of five sample (text, input-token) pairs spanning mechanical, reasoning, and long-context cases.
- The loop routes each query, prints the chosen tier, and accumulates `routed` cost; `front` totals the all-frontier cost for the same batch.
- **In one line:** free, instant routing on surface signals that catches the obvious cases but is brittle on the tricky ones.

**What you'll see:** Each query mapped to a tier (classify/extract/tag -> small, design -> mid, the 4000-token contract -> frontier), then a batch cost about 1.6x cheaper than all-frontier, decided before any call.

## 4 - Semantic / classifier routing

This cell routes on meaning instead of surface signals, so it catches the short-but-hard query that a length rule would misroute. A tiny difficulty classifier scores each query's content and maps the score to a tier.

In [ ]:
TIERS = {                       # illustrative $/1M tokens + eval-score; NOT live prices
    "frontier": {"in": 5.00, "out": 25.00, "q": 0.95},   # opus / GPT-5-class
    "mid":      {"in": 3.00, "out": 15.00, "q": 0.90},   # sonnet-class
    "small":    {"in": 1.00, "out":  5.00, "q": 0.82},   # haiku-class
    "nano":     {"in": 0.25, "out":  1.50, "q": 0.72},   # flash-lite / nano-class
}
def cost(tier, tin, tout):      # per-request cost in USD
    return (tin * TIERS[tier]["in"] + tout * TIERS[tier]["out"]) / 1_000_000
# Semantic / classifier routing (~5-100 ms): score the query's DIFFICULTY from features a small model
# reads, then route by threshold. Catches hard queries that the length/keyword rules miss.
TIN, TOUT = 800, 300
def difficulty(query):                 # a tiny bag-of-signals classifier (stand-in for an embedding model)
    q = query.lower(); score = 0.30
    for kw, w in [("prove",0.4),("why",0.25),("trade-off",0.3),("edge case",0.35),
                  ("step by step",0.3),("ambiguous",0.3),("nuance",0.3)]:
        if kw in q: score += w
    return min(score, 1.0)
def route_semantic(query):
    d = difficulty(query)
    return ("frontier" if d >= 0.7 else "mid" if d >= 0.5 else "small"), d
cases = [
    "Prove why this distributed lock is correct under a network partition edge case",  # SHORT but HARD
    "What is the capital of France",
    "Explain the nuance in this ambiguous requirement step by step",
]
print("Semantic routing (~50 ms classifier vs 500-2000 ms model response = negligible):")
for text in cases:
    t, d = route_semantic(text)
    print("  difficulty {:.2f} -> {:<9} <- {}".format(d, t, text[:42]))
print()
print("A rule on LENGTH alone sends the 12-word proof query to 'small' and gets a wrong answer;")
print("the difficulty score reads the CONTENT and lifts it to frontier. Classifiers generalize past brittle rules.")

# Output:
# Semantic routing (~50 ms classifier vs 500-2000 ms model response = negligible):
#   difficulty 1.00 -> frontier  <- Prove why this distributed lock is correct
#   difficulty 0.30 -> small     <- What is the capital of France
#   difficulty 1.00 -> frontier  <- Explain the nuance in this ambiguous requi
#
# A rule on LENGTH alone sends the 12-word proof query to 'small' and gets a wrong answer;
# the difficulty score reads the CONTENT and lifts it to frontier. Classifiers generalize past brittle rules.

**Explanation**

A content-aware difficulty router - a stand-in for an embedding model or learned scorer. `difficulty()` accumulates weight from difficulty-signaling phrases in the query, and `route_semantic()` maps the resulting score to a tier by threshold. It demonstrates lifting a 12-word proof query to the frontier where a length rule would have failed.

**How the code works - step by step**
- `difficulty(query)` starts at a 0.30 baseline and adds weight for signals like 'prove', 'why', 'trade-off', 'edge case', 'nuance', capping at 1.0.
- `route_semantic(query)` returns frontier for score >= 0.7, mid for >= 0.5, else small, plus the raw score.
- `cases` includes a short-but-hard concurrency proof, an easy factual question, and an ambiguous-nuance request.
- The loop prints each query's difficulty score and routed tier, then contrasts it with what a length-only rule would have done.
- **In one line:** routing on content, not length, catches hard queries rules miss for a negligible latency cost.

**What you'll see:** The short 12-word proof scores 1.00 and is routed to frontier, the capital-of-France query scores 0.30 -> small, and the nuance request scores 1.00 -> frontier - proving the classifier catches what a length rule misroutes.

## 5 - Cascade routing

This cell implements the FrugalGPT cheap-first cascade: run the small model on every query, read a confidence signal, and escalate to the frontier only when confidence is low. It also surfaces the double-pay trap that can make a cascade cost more than all-frontier.

In [ ]:
TIERS = {                       # illustrative $/1M tokens + eval-score; NOT live prices
    "frontier": {"in": 5.00, "out": 25.00, "q": 0.95},   # opus / GPT-5-class
    "mid":      {"in": 3.00, "out": 15.00, "q": 0.90},   # sonnet-class
    "small":    {"in": 1.00, "out":  5.00, "q": 0.82},   # haiku-class
    "nano":     {"in": 0.25, "out":  1.50, "q": 0.72},   # flash-lite / nano-class
}
def cost(tier, tin, tout):      # per-request cost in USD
    return (tin * TIERS[tier]["in"] + tout * TIERS[tier]["out"]) / 1_000_000
# Cascade routing (FrugalGPT): run the CHEAP model first, read a confidence signal, and ESCALATE to a
# strong model ONLY when confidence is low. Most accurate per dollar - but escalated queries pay TWICE.
CONF_THRESHOLD = 0.70
TIN, TOUT, N = 900, 300, 100
# illustrative per-query confidence from the small model; the ~25% below the threshold escalate
low  = [0.52, 0.58, 0.63, 0.67, 0.69] * 5     # 25 low-confidence  -> escalate to frontier
high = [0.74, 0.80, 0.86, 0.91, 0.96] * 15    # 75 high-confidence -> stop at small
confidences = (low + high)[:N]
small_c, front_c = cost("small", TIN, TOUT), cost("frontier", TIN, TOUT)
escalated = sum(1 for c in confidences if c < CONF_THRESHOLD)
# every query pays the small model; the low-confidence ones ALSO pay the frontier model
cascade_cost = N * small_c + escalated * front_c
all_front = N * front_c
print("Cascade over {} queries, escalate when confidence < {:.2f}:".format(N, CONF_THRESHOLD))
print("  escalated: {}/{} queries ({:.0f}%) went small -> frontier".format(escalated, N, escalated/N*100))
print("  cost: cascade ${:.4f}  vs all-frontier ${:.4f}  = {:.1f}x cheaper".format(cascade_cost, all_front, all_front/cascade_cost))
print("  the {} hard queries STILL got the frontier answer - only the easy {} stopped at small.".format(escalated, N-escalated))
print()
print("Watch the double-pay: an escalated query costs small + frontier (${:.4f}). If you escalate too often,".format(small_c+front_c))
print("cascade costs MORE than all-frontier. The confidence THRESHOLD is the knob: higher = safer + pricier.")

# Output:
# Cascade over 100 queries, escalate when confidence < 0.70:
#   escalated: 25/100 queries (25%) went small -> frontier
#   cost: cascade $0.5400  vs all-frontier $1.2000  = 2.2x cheaper
#   the 25 hard queries STILL got the frontier answer - only the easy 75 stopped at small.
#
# Watch the double-pay: an escalated query costs small + frontier ($0.0144). If you escalate too often,
# cascade costs MORE than all-frontier. The confidence THRESHOLD is the knob: higher = safer + pricier.

**Explanation**

A cascade cost simulator. Every query pays the small model; the low-confidence subset (below `CONF_THRESHOLD`) also pays the frontier, so escalated queries pay twice. It counts escalations over 100 queries and compares total cost to the all-frontier baseline, spotlighting the confidence threshold as the governing knob.

**How the code works - step by step**
- `CONF_THRESHOLD = 0.70` is the escalation cutoff; `low` and `high` build a confidence distribution where ~25% fall below it.
- `escalated` counts queries under the threshold; `cascade_cost = N*small_c + escalated*front_c` charges every query the small model plus the frontier for escalations.
- `all_front` is the all-frontier baseline for the same 100 queries.
- It prints the escalation rate, the cost comparison, and the per-escalated-query double-pay to explain the break-even risk.
- **In one line:** cheap-first with confidence-gated escalation is most accurate per dollar until too many escalations flip the saving.

**What you'll see:** 25/100 queries escalate; the cascade costs ~$0.54 vs ~$1.20 all-frontier (about 2.2x cheaper), with a note that each escalated query pays small+frontier so too high an escalation rate reverses the win.

## 6 - The routing gateway + fallback

This cell puts routing where it belongs in production - the AI gateway - with a route table plus a per-model fallback chain, so a provider error, rate-limit, or context overflow degrades to another model instead of returning a 500.

In [ ]:
TIERS = {                       # illustrative $/1M tokens + eval-score; NOT live prices
    "frontier": {"in": 5.00, "out": 25.00, "q": 0.95},   # opus / GPT-5-class
    "mid":      {"in": 3.00, "out": 15.00, "q": 0.90},   # sonnet-class
    "small":    {"in": 1.00, "out":  5.00, "q": 0.82},   # haiku-class
    "nano":     {"in": 0.25, "out":  1.50, "q": 0.72},   # flash-lite / nano-class
}
def cost(tier, tin, tout):      # per-request cost in USD
    return (tin * TIERS[tier]["in"] + tout * TIERS[tier]["out"]) / 1_000_000
# The routing gateway (Lesson 12.3) is where routing LIVES in production: a route table plus a fallback
# chain per model, so a provider error / rate-limit / context-overflow degrades to a slower answer, not a 500.
route_table = {"classify": "small", "summarize": "mid", "reason": "frontier"}
fallbacks = {"small": ["mid", "frontier"], "mid": ["frontier"], "frontier": ["mid"]}   # try in order on failure
def serve(task, failing):                 # failing = set of tiers currently erroring / rate-limited
    chain = [route_table[task]] + fallbacks[route_table[task]]
    for tier in chain:
        if tier not in failing:
            return tier, chain.index(tier)
    return None, len(chain)
requests = [("classify", set()), ("summarize", {"mid"}), ("reason", {"frontier"}), ("classify", {"small","mid"})]
print("Gateway route table + fallback chain (degrade, do not drop):")
for task, failing in requests:
    tier, hops = serve(task, failing)
    note = "primary" if hops == 0 else "fallback after {} hop(s)".format(hops)
    fail_note = " (failing: {})".format(sorted(failing)) if failing else ""
    print("  {:<10} -> {:<9} [{}]{}".format(task, tier, note, fail_note))
print()
print("LiteLLM ships 3 fallback types: general, context_window, and content_policy.")
print("Routing is a GATEWAY concern: one place decides the model, retries on failure, and meters cost per route.")

# Output:
# Gateway route table + fallback chain (degrade, do not drop):
#   classify   -> small     [primary]
#   summarize  -> frontier  [fallback after 1 hop(s)] (failing: ['mid'])
#   reason     -> mid       [fallback after 1 hop(s)] (failing: ['frontier'])
#   classify   -> frontier  [fallback after 2 hop(s)] (failing: ['mid', 'small'])
#
# LiteLLM ships 3 fallback types: general, context_window, and content_policy.
# Routing is a GATEWAY concern: one place decides the model, retries on failure, and meters cost per route.

**Explanation**

A gateway route-table simulator. `route_table` maps each task to a primary tier and `fallbacks` gives each tier an ordered backup chain. `serve()` walks the chain past any currently-failing tiers and returns the first healthy one plus how many hops it took, modeling graceful degradation rather than dropped requests.

**How the code works - step by step**
- `route_table` maps tasks (classify/summarize/reason) to primary tiers; `fallbacks` lists the ordered backup chain per tier.
- `serve(task, failing)` builds `[primary] + fallbacks` and returns the first tier not in the `failing` set, along with the hop count.
- `requests` simulates four scenarios, including tiers being down (rate-limited/erroring).
- The loop prints the served tier, whether it was primary or a fallback (with hop count), and which tiers were failing.
- **In one line:** the gateway is the one place that picks the model, retries on failure, and meters cost per route.

**What you'll see:** Each request resolves to a healthy tier: classify -> small (primary), summarize -> frontier after the mid tier fails (1 hop), reason -> mid after frontier fails, and classify -> frontier after 2 hops - degrade, never drop.

## 7 - Measure the router

This final cell treats the router as a bet: A/B it against the all-frontier baseline on cost AND quality, keep it only if quality holds, and watch each route for silent drift when a model quietly degrades upstream.

In [ ]:
TIERS = {                       # illustrative $/1M tokens + eval-score; NOT live prices
    "frontier": {"in": 5.00, "out": 25.00, "q": 0.95},   # opus / GPT-5-class
    "mid":      {"in": 3.00, "out": 15.00, "q": 0.90},   # sonnet-class
    "small":    {"in": 1.00, "out":  5.00, "q": 0.82},   # haiku-class
    "nano":     {"in": 0.25, "out":  1.50, "q": 0.72},   # flash-lite / nano-class
}
def cost(tier, tin, tout):      # per-request cost in USD
    return (tin * TIERS[tier]["in"] + tout * TIERS[tier]["out"]) / 1_000_000
# Never route blind: a router is a BET. A/B it against the all-frontier baseline on cost AND quality,
# keep it only if quality holds, and watch EACH route for silent drift (a model that quietly got worse).
QUALITY_FLOOR = 0.88
TIN, TOUT, REQ = 1000, 300, 10000
mix = {"small": 0.70, "mid": 0.20, "frontier": 0.10}   # traffic share per route
def weighted(quality_by_tier):
    return sum(mix[t] * quality_by_tier[t] for t in mix)
router_cost = sum(mix[t] * cost(t, TIN, TOUT) for t in mix) * REQ
base_cost   = cost("frontier", TIN, TOUT) * REQ
healthy = {"small": 0.88, "mid": 0.92, "frontier": 0.96}   # per-route quality, measured on the traffic each gets
rq = weighted(healthy)
verdict = "KEEP the router" if rq >= QUALITY_FLOOR else "REJECT (quality below floor)"
print("A/B the router vs all-frontier (quality floor = {:.2f}):".format(QUALITY_FLOOR))
print("  router:       ${:,.2f}/day  quality {:.3f}".format(router_cost, rq))
print("  all-frontier: ${:,.2f}/day  quality {:.3f}".format(base_cost, healthy["frontier"]))
print("  -> {:.0f}% cheaper, quality {:.3f} vs {:.2f} floor -> {}.".format((1-router_cost/base_cost)*100, rq, QUALITY_FLOOR, verdict))
print()
# Silent drift: the 'small' route's model is deprecated / quietly degrades from 0.88 -> 0.80
drifted = dict(healthy); drifted["small"] = 0.80
dq = weighted(drifted)
alert = "floor BREACHED -> alert + reroute" if dq < QUALITY_FLOOR else "still above floor"
print("Drift watch: the 'small' route quietly drops 0.88 -> 0.80 (a model change upstream).")
print("  weighted router quality falls {:.3f} -> {:.3f}  vs {:.2f} floor -> {}.".format(rq, dq, QUALITY_FLOOR, alert))
print("Track cost saved AND per-route quality. A bad route degrades silently while the bill still looks great.")

# Output:
# A/B the router vs all-frontier (quality floor = 0.88):
#   router:       $45.00/day  quality 0.896
#   all-frontier: $125.00/day  quality 0.960
#   -> 64% cheaper, quality 0.896 vs 0.88 floor -> KEEP the router.
#
# Drift watch: the 'small' route quietly drops 0.88 -> 0.80 (a model change upstream).
#   weighted router quality falls 0.896 -> 0.840  vs 0.88 floor -> floor BREACHED -> alert + reroute.
# Track cost saved AND per-route quality. A bad route degrades silently while the bill still looks great.

**Explanation**

A router A/B-plus-drift monitor. It computes the router's weighted cost and quality from a traffic mix, compares against the all-frontier baseline and a `QUALITY_FLOOR`, then simulates one route's model quietly degrading and shows the weighted quality breaching the floor even while cost stays flat.

**How the code works - step by step**
- `QUALITY_FLOOR = 0.88` and `mix` (70% small / 20% mid / 10% frontier) define the acceptance bar and traffic shares.
- `weighted()` computes traffic-weighted quality; `router_cost` and `base_cost` give the router's vs all-frontier daily cost.
- With `healthy` per-route scores it prints the A/B verdict (keep vs reject) - the router is 64% cheaper and above the floor, so KEEP.
- It then drifts the small route 0.88 -> 0.80, recomputes weighted quality, and fires an alert because the floor is breached while the bill is unchanged.
- **In one line:** track cost saved AND per-route quality, because a bad route degrades silently while the bill still looks great.

**What you'll see:** The router shows $45/day at quality 0.896 vs $125/day at 0.960 all-frontier -> 64% cheaper, KEEP. After the small route drifts, weighted quality falls 0.896 -> 0.840, breaching the 0.88 floor and triggering an alert + reroute.

You now have the full routing pipeline in miniature: price the spread, select the cheapest model that clears the bar, route the obvious cases with free rules and the ambiguous ones with a classifier, escalate only the hard queries with a cascade, serve it all through a gateway with fallbacks, and - the step everyone skips - measure each route so a silently drifting one can't wreck quality while the bill still looks great. Ask any router two questions: did it cut cost meaningfully, and did quality hold on every route? Latency (13.4) and self-hosting a route target (13.5/13.6) build on top of this.